# 04 Feature Engineering

**Owner**: Zeshen Zhang (zz10740)

Takes `flights_clean` + `weather_clean` and produces the final feature dataset used by analysis and modeling.

Pipeline:
1. **Timezone alignment** — convert flight local `CRSDepTime` to UTC using a 24-hub IATA→IANA map
2. **Hourly weather aggregation** — collapse multiple METAR observations per hour to one row
3. **Weather join** — left join on `(origin, UTC date, UTC hour)` so the weather signal matches the actual departure hour at the airport
4. **Null-safe derived weather features** — `is_low_visibility`, `is_high_wind`, `has_precipitation`, plus a `has_weather_data` flag so downstream can decide whether to filter or impute
5. **Tail Number Tracking (Ripple Effect)** — window over `(Tail_Number, FlightDate)` to derive `flight_leg`, `prev_arr_delay`, `inherited_delay`, `cumulative_delay`, `delay_recovery`
6. **Historical features** — route / carrier / origin-airport averages computed from 2019–2023 only (to avoid leakage for 2024 predictions)

Writes:
- `/data/processed/flights_with_weather/` (checkpoint)
- `/data/processed/features/final/`       (main output)


## 1. Spark session

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import FloatType, IntegerType, StringType

spark = (
    SparkSession.builder
    .appName("Feature-Engineering")
    # 200 shuffle partitions (vs Kaiyang's ETL default of 8) — this pipeline does
    # window functions + joins + a 72-way partitioned write, so smaller partitions
    # prevent OOM when writing flights_with_weather.
    .config("spark.sql.shuffle.partitions", "200")
    # 6g heap (not 8g): leaves headroom for JVM off-heap + Snappy buffers + Python
    # so the total process footprint stays under the Docker container's memory limit.
    .config("spark.driver.memory", "6g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

Spark 3.5.0


## 2. Load data

In [2]:
FLIGHTS_PATH = "/data/processed/flights_clean"
WEATHER_PATH = "/data/processed/weather_clean"
OUTPUT_FWW   = "/data/processed/flights_with_weather"
OUTPUT_FINAL = "/data/processed/features/final"

flights = spark.read.parquet(FLIGHTS_PATH)
weather = spark.read.parquet(WEATHER_PATH)
print(f"flights : {flights.count():>12,} rows  /  {len(flights.columns):>2} cols")
print(f"weather : {weather.count():>12,} rows  /  {len(weather.columns):>2} cols")

flights :   37,786,688 rows  /  27 cols
weather :    1,474,038 rows  /  17 cols


## 3. Airport → timezone map

Weather `obs_time` is UTC. Flight `CRSDepTime` is *local* time at the origin airport.
To align the two we need each flight's true UTC departure hour.

The 24 hubs covered by the weather dataset span 5 IANA zones. Hardcoding the map is cleaner
than using `timezonefinder` (and correctly handles Arizona, which doesn't observe DST — `PHX`
uses `America/Phoenix`, not `America/Denver`).

In [3]:
AIRPORT_TZ = [
    ("ATL", "America/New_York"),
    ("AUS", "America/Chicago"),
    ("BOS", "America/New_York"),
    ("CLT", "America/New_York"),
    ("DEN", "America/Denver"),
    ("DFW", "America/Chicago"),
    ("DTW", "America/Detroit"),     # same UTC offset as New_York but named separately
    ("EWR", "America/New_York"),
    ("FLL", "America/New_York"),
    ("IAH", "America/Chicago"),
    ("JFK", "America/New_York"),
    ("LAS", "America/Los_Angeles"),
    ("LAX", "America/Los_Angeles"),
    ("LGA", "America/New_York"),
    ("MCO", "America/New_York"),
    ("MDW", "America/Chicago"),
    ("MIA", "America/New_York"),
    ("MSP", "America/Chicago"),
    ("ORD", "America/Chicago"),
    ("PHL", "America/New_York"),
    ("PHX", "America/Phoenix"),     # no DST
    ("SAN", "America/Los_Angeles"),
    ("SEA", "America/Los_Angeles"),
    ("SFO", "America/Los_Angeles"),
]
HUB_CODES = {iata for iata, _ in AIRPORT_TZ}
tz_df = spark.createDataFrame(AIRPORT_TZ, ["origin_iata", "origin_tz"])
tz_df.show(24, truncate=False)

+-----------+-------------------+
|origin_iata|origin_tz          |
+-----------+-------------------+
|ATL        |America/New_York   |
|AUS        |America/Chicago    |
|BOS        |America/New_York   |
|CLT        |America/New_York   |
|DEN        |America/Denver     |
|DFW        |America/Chicago    |
|DTW        |America/Detroit    |
|EWR        |America/New_York   |
|FLL        |America/New_York   |
|IAH        |America/Chicago    |
|JFK        |America/New_York   |
|LAS        |America/Los_Angeles|
|LAX        |America/Los_Angeles|
|LGA        |America/New_York   |
|MCO        |America/New_York   |
|MDW        |America/Chicago    |
|MIA        |America/New_York   |
|MSP        |America/Chicago    |
|ORD        |America/Chicago    |
|PHL        |America/New_York   |
|PHX        |America/Phoenix    |
|SAN        |America/Los_Angeles|
|SEA        |America/Los_Angeles|
|SFO        |America/Los_Angeles|
+-----------+-------------------+



## 4. Compute UTC departure timestamp

`CRSDepTime` is an integer in `HHMM` form (e.g. `1430` → 14:30). Two edge cases:

- values `1`–`59` mean `00:01`–`00:59` → need left-pad
- `2400` means 24:00, i.e. midnight next day → roll the date forward and clamp hour to 0

`to_utc_timestamp(ts, tz)` interprets `ts` as local-in-`tz` and returns the UTC timestamp.

In [4]:
flights_tz = flights.join(
    F.broadcast(tz_df),
    flights.Origin == tz_df.origin_iata,
    "left",
).drop("origin_iata")

flights_tz = (
    flights_tz
    .withColumn("_h_raw", (F.col("CRSDepTime") / 100).cast(IntegerType()))
    .withColumn("_m",     (F.col("CRSDepTime") % 100).cast(IntegerType()))
    .withColumn("_date_adj",
        F.when(F.col("_h_raw") >= 24, F.date_add("FlightDate", 1))
         .otherwise(F.col("FlightDate")))
    .withColumn("_h", F.when(F.col("_h_raw") >= 24, 0).otherwise(F.col("_h_raw")))
    .withColumn("_local_ts_str",
        F.concat(
            F.date_format("_date_adj", "yyyy-MM-dd"), F.lit(" "),
            F.lpad(F.col("_h").cast("string"), 2, "0"), F.lit(":"),
            F.lpad(F.col("_m").cast("string"), 2, "0"), F.lit(":00"),
        ))
    .withColumn("dep_utc_ts",   F.expr("to_utc_timestamp(_local_ts_str, origin_tz)"))
    .withColumn("dep_utc_date", F.to_date("dep_utc_ts"))
    .withColumn("dep_utc_hour", F.hour("dep_utc_ts"))
    .drop("_h_raw", "_m", "_date_adj", "_h", "_local_ts_str")
)

flights_tz.select(
    "Origin", "FlightDate", "CRSDepTime", "origin_tz",
    "dep_utc_ts", "dep_utc_date", "dep_utc_hour",
).where(F.col("Origin").isin("JFK", "LAX", "PHX", "ORD")).show(8, truncate=False)

+------+----------+----------+---------------+-------------------+------------+------------+
|Origin|FlightDate|CRSDepTime|origin_tz      |dep_utc_ts         |dep_utc_date|dep_utc_hour|
+------+----------+----------+---------------+-------------------+------------+------------+
|ORD   |2024-05-02|740       |America/Chicago|2024-05-02 12:40:00|2024-05-02  |12          |
|ORD   |2024-05-03|745       |America/Chicago|2024-05-03 12:45:00|2024-05-03  |12          |
|ORD   |2024-05-05|740       |America/Chicago|2024-05-05 12:40:00|2024-05-05  |12          |
|ORD   |2024-05-06|745       |America/Chicago|2024-05-06 12:45:00|2024-05-06  |12          |
|ORD   |2024-05-09|735       |America/Chicago|2024-05-09 12:35:00|2024-05-09  |12          |
|ORD   |2024-05-10|735       |America/Chicago|2024-05-10 12:35:00|2024-05-10  |12          |
|ORD   |2024-05-12|735       |America/Chicago|2024-05-12 12:35:00|2024-05-12  |12          |
|ORD   |2024-05-13|735       |America/Chicago|2024-05-13 12:35:00|2024

## 5. Hourly weather aggregation

Weather stations often report multiple METAR observations within the same hour. Collapse to one
row per `(iata_code, obs_date, obs_hour)` by averaging numeric fields and union-ing weather codes.

In [5]:
weather_hourly = (
    weather.groupBy("iata_code", "obs_date", "obs_hour")
    .agg(
        F.avg("tmpf").alias("temperature"),
        F.avg("dwpf").alias("dewpoint"),
        F.avg("relh").alias("humidity"),
        F.avg("drct").alias("wind_direction"),
        F.avg("sknt").alias("wind_speed"),
        F.avg("vsby").alias("visibility"),
        F.avg("p01i").alias("precipitation"),
        F.concat_ws(
            ",",
            F.collect_set(F.when(F.col("wxcodes").isNotNull(), F.col("wxcodes"))),
        ).alias("weather_codes"),
    )
)
print(f"weather_hourly: {weather_hourly.count():,} rows")

weather_hourly: 1,261,105 rows


## 6. Join flights ← weather (UTC-aligned)

Left join so every flight is kept. Flights whose origin isn't one of the 24 hubs will get null
weather columns — that's expected and tracked via the `has_weather_data` flag below.

In [6]:
# No F.broadcast() hint: weather_hourly is ~1.2M rows / ~200MB, too big to broadcast safely
# in local mode. A sort-merge join with 200 shuffle partitions is reliable.
fw = flights_tz.join(
    weather_hourly,
    (flights_tz.Origin       == weather_hourly.iata_code) &
    (flights_tz.dep_utc_date == weather_hourly.obs_date) &
    (flights_tz.dep_utc_hour == weather_hourly.obs_hour),
    "left",
).drop("iata_code", "obs_date", "obs_hour")

fw = (
    fw
    .withColumn("has_weather_data",
        F.when(F.col("temperature").isNotNull(), 1).otherwise(0))
    # Null-safe: null stays null so downstream can tell "unknown" from "known-clear"
    .withColumn("is_low_visibility",
        F.when(F.col("visibility").isNull(), None)
         .when(F.col("visibility") < 3, 1).otherwise(0))
    .withColumn("is_high_wind",
        F.when(F.col("wind_speed").isNull(), None)
         .when(F.col("wind_speed") > 25, 1).otherwise(0))
    .withColumn("has_precipitation",
        F.when(F.col("weather_codes").isNull() | (F.col("weather_codes") == ""), None)
         .when(F.col("weather_codes").rlike("RA|SN|TS|DZ|FZ|GR|GS"), 1).otherwise(0))
)

### Sanity check: weather match rate

Should be close to 100% on hub-origin flights (≥99%). Anything lower hints at a timezone
bug somewhere above.

In [7]:
total       = fw.count()
matched     = fw.filter(F.col("has_weather_data") == 1).count()
hub_total   = fw.filter(F.col("Origin").isin(*HUB_CODES)).count()
hub_matched = fw.filter(F.col("Origin").isin(*HUB_CODES) & (F.col("has_weather_data") == 1)).count()

print(f"All flights       : matched {matched:>11,} / {total:>11,}  ({matched/total*100:5.1f}%)")
print(f"Hub-origin flights: matched {hub_matched:>11,} / {hub_total:>11,}  ({hub_matched/hub_total*100:5.1f}%)")

All flights       : matched  21,232,724 /  37,786,688  ( 56.2%)
Hub-origin flights: matched  21,232,724 /  21,261,806  ( 99.9%)


## 7. Checkpoint: save `flights_with_weather/`

In [8]:
(
    fw.write
      .partitionBy("Year", "Month")
      .mode("overwrite")
      .parquet(OUTPUT_FWW)
)
print(f"Wrote: {OUTPUT_FWW}")

Wrote: /data/processed/flights_with_weather


## 8. Tail Number Tracking (Ripple Effect)

For each aircraft, order its flights within a day by scheduled departure and derive:

- `flight_leg` — 1-indexed segment number
- `prev_arr_delay` / `prev_origin` / `prev_dest` — previous segment's arrival delay and route
- `inherited_delay` — positive prev delay carried into this segment (0 otherwise)
- `cumulative_delay` — sum of `ArrDelay` across all segments so far today
- `delay_recovery` — how much delay this segment absorbed (prev − current)

Flights with a null `Tail_Number` can't participate — they get null ripple features but
stay in the dataset.

In [9]:
df = spark.read.parquet(OUTPUT_FWW)

with_tail = df.filter(F.col("Tail_Number").isNotNull() & (F.col("Tail_Number") != ""))
no_tail   = df.filter(F.col("Tail_Number").isNull()   | (F.col("Tail_Number") == ""))

w = Window.partitionBy("Tail_Number", "FlightDate").orderBy("CRSDepTime")

with_tail = (
    with_tail
    .withColumn("flight_leg",     F.row_number().over(w))
    .withColumn("prev_arr_delay", F.lag("ArrDelay", 1).over(w))
    .withColumn("prev_origin",    F.lag("Origin",   1).over(w))
    .withColumn("prev_dest",      F.lag("Dest",     1).over(w))
    .withColumn("cumulative_delay",
        F.sum("ArrDelay").over(w.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
    .withColumn("inherited_delay",
        F.when((F.col("flight_leg") > 1) & (F.col("prev_arr_delay") > 0),
               F.col("prev_arr_delay")).otherwise(F.lit(0.0).cast(FloatType())))
    .withColumn("delay_recovery",
        F.when(F.col("flight_leg") > 1,
               F.col("prev_arr_delay") - F.col("ArrDelay")).otherwise(None))
)

ripple_cols = [
    ("flight_leg",      IntegerType()),
    ("prev_arr_delay",  FloatType()),
    ("prev_origin",     StringType()),
    ("prev_dest",       StringType()),
    ("cumulative_delay", FloatType()),
    ("inherited_delay", FloatType()),
    ("delay_recovery",  FloatType()),
]
for name, ty in ripple_cols:
    no_tail = no_tail.withColumn(name, F.lit(None).cast(ty))

df = with_tail.unionByName(no_tail)
print(f"After ripple features: {df.count():,} rows, {len(df.columns)} cols")

After ripple features: 37,786,688 rows, 50 cols


### Sanity check: trace one aircraft's day

In [10]:
# Pick a tail number with 4+ legs on a single day and show the propagation pattern
sample_key = (
    df.filter(F.col("flight_leg") >= 4)
      .select("Tail_Number", "FlightDate")
      .limit(1)
      .collect()
)
if sample_key:
    tn, fd = sample_key[0].Tail_Number, sample_key[0].FlightDate
    print(f"Aircraft {tn} on {fd}:")
    (df.filter((F.col("Tail_Number") == tn) & (F.col("FlightDate") == fd))
       .orderBy("CRSDepTime")
       .select("flight_leg", "Origin", "Dest", "CRSDepTime",
               "DepDelay", "ArrDelay", "prev_arr_delay",
               "inherited_delay", "cumulative_delay", "delay_recovery")
       .show(truncate=False))

Aircraft 188NV on 2024-02-11:
+----------+------+----+----------+--------+--------+--------------+---------------+----------------+--------------+
|flight_leg|Origin|Dest|CRSDepTime|DepDelay|ArrDelay|prev_arr_delay|inherited_delay|cumulative_delay|delay_recovery|
+----------+------+----+----------+--------+--------+--------------+---------------+----------------+--------------+
|1         |SFB   |USA |710       |-14.0   |-20.0   |NULL          |0.0            |-20.0           |NULL          |
|2         |USA   |SFB |933       |-9.0    |-7.0    |-20.0         |0.0            |-27.0           |-13.0         |
|3         |SFB   |GSP |1154      |-9.0    |-10.0   |-7.0          |0.0            |-37.0           |3.0           |
|4         |GSP   |SFB |1409      |62.0    |53.0    |-10.0         |0.0            |16.0            |-63.0         |
+----------+------+----+----------+--------+--------+--------------+---------------+----------------+--------------+



## 9. Historical features

Route / carrier / origin-airport aggregates for predictive features. Computed on **2019–2023**
only so that 2024 predictions don't leak future information. For 2019–2023 flights this does
include their own row in the aggregate, but the dilution across millions of observations makes
this negligible in practice.

In [11]:
hist = df.filter(F.col("Year") <= 2023)

route_stats = (
    hist.groupBy("Origin", "Dest")
        .agg(
            F.count("*").alias("route_total_flights"),
            F.avg("ArrDelay").alias("route_avg_delay"),
            F.avg(F.when(F.col("ArrDelay") <= 15, 1.0).otherwise(0.0)).alias("route_on_time_rate"),
        )
)
carrier_stats = (
    hist.groupBy("Reporting_Airline")
        .agg(
            F.avg("ArrDelay").alias("carrier_avg_delay"),
            F.avg(F.when(F.col("ArrDelay") <= 15, 1.0).otherwise(0.0)).alias("carrier_on_time_rate"),
        )
)
origin_stats = (
    hist.groupBy("Origin")
        .agg(
            F.avg("DepDelay").alias("origin_avg_dep_delay"),
            F.stddev("DepDelay").alias("origin_std_dep_delay"),
        )
)

df = (
    df
    .join(F.broadcast(route_stats),   ["Origin", "Dest"],      "left")
    .join(F.broadcast(carrier_stats), ["Reporting_Airline"],   "left")
    .join(F.broadcast(origin_stats),  ["Origin"],              "left")
)
print(f"After historical features: {len(df.columns)} cols")

After historical features: 57 cols


## 10. Save final feature dataset

In [12]:
print("Final schema:")
df.printSchema()

(
    df.write
      .partitionBy("Year", "Month")
      .mode("overwrite")
      .parquet(OUTPUT_FINAL)
)
print(f"\nWrote: {OUTPUT_FINAL}")

Final schema:
root
 |-- Origin: string (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Tail_Number: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: integer (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- DestCityName: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- DepTime: integer (nullable = true)
 |-- DepDelay: float (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- ArrTime: integer (nullable = true)
 |-- ArrDelay: float (nullable = true)
 |-- Cancelled: integer (nullable = true)
 |-- CancellationCode: string (nullable = true)
 |-- Diverted: integer (nullable = true)
 |-- Distance: float (nullable = true)
 |-- CarrierDelay: float (nullable = true)
 |-- WeatherDelay: float (nullable = true)
 |-- NASDelay: float (nullable = true)
 |-- SecurityDelay: float (nullable = true)
 |-- LateAircraftDelay: float (nullabl

## 11. Final verification

In [13]:
final_df = spark.read.parquet(OUTPUT_FINAL)
print(f"final rows : {final_df.count():,}")
print(f"final cols : {len(final_df.columns)}")
print()
print("Weather coverage:")
final_df.groupBy("has_weather_data").count().orderBy("has_weather_data").show()

print("Ripple feature coverage (non-null flight_leg):")
have_ripple = final_df.filter(F.col("flight_leg").isNotNull()).count()
print(f"  {have_ripple:,} / {final_df.count():,}  ({have_ripple/final_df.count()*100:.1f}%)")

spark.stop()

final rows : 37,786,688
final cols : 57

Weather coverage:
+----------------+--------+
|has_weather_data|   count|
+----------------+--------+
|               0|16553964|
|               1|21232724|
+----------------+--------+

Ripple feature coverage (non-null flight_leg):
  37,786,688 / 37,786,688  (100.0%)
